In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import load_img

# pre-trained model packages
from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.applications.vgg16 import decode_predictions
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
path = '/content/clothing-dataset-small-master/train/t-shirt'
name = '5f0a3fa0-6a3d-4b68-b213-72766a643de7.jpg'
fullname = f'{path}/{name}'
img = load_img(fullname, target_size=(224, 224, 3))
img

# Pre-trained convolutional neural networks
- Imagenet dataset: https://www.image-net.org/
- Pre-trained models: https://keras.io/api/applications/

In [ ]:
model = VGG16(weights='imagenet', input_shape=(224, 224, 3))

In [ ]:
x = np.array(img)
print(f'one image shape {x.shape}')

X = np.array([x])
print(f'one batch image shape {X.shape}')

# Preprocessing
X = preprocess_input(X)

print(X)

In [ ]:
pred = model.predict(X)
decode_predictions(pred) # this what the model is  before fitting our data on it

# Transfer learning
- Reading data with **ImageDataGenerator**
- Train **VGG16** on smaller images (150x150)

In [ ]:
train_path = '/content/clothing-dataset-small-master/train'

train_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

train_ds = train_gen.flow_from_directory(
    train_path,
    target_size=(150, 150),
    batch_size=32
)

In [ ]:
train_ds.class_indices

In [ ]:
val_path = '/content/clothing-dataset-small-master/validation'

val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_ds = val_gen.flow_from_directory(
    val_path,
    target_size=(150, 150),
    batch_size=32,
    shuffle=False
)

In [ ]:
def base_model():
    base_model = VGG16(
        include_top=False,
        weights='imagenet',
        input_shape=(150, 150, 3)
    )

    base_model.trainable = False

    inputs = keras.Input(shape=(150, 150, 3))

    base = base_model(inputs, training=False)

    vectors = keras.layers.GlobalAveragePooling2D()(base)

    outputs = keras.layers.Dense(10, activation='softmax')(vectors)

    model = keras.Model(inputs, outputs)

    return model

In [ ]:
learning_rate = 0.01

optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

loss = keras.losses.CategoricalCrossentropy()

model = base_model()
model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])

In [ ]:
res = model.fit(train_ds, epochs=10, validation_data=val_ds)

In [ ]:
plt.plot(res.history['accuracy'], label='train_acc')
plt.plot(res.history['val_accuracy'], label='val_acc')
plt.xticks(np.arange(10))
plt.legend()

# Data augmentation

In [ ]:
train_gen = ImageDataGenerator(

    preprocessing_function=preprocess_input,  # Apply Xception preprocessing

    rotation_range=30,        # Randomly rotate images up to 30 degrees

    width_shift_range=0.1,    # Random horizontal shift (10% of width)

    height_shift_range=0.1,   # Random vertical shift (10% of height)

    shear_range=0.1,          # Shear transformation

    zoom_range=0.2,           # Random zoom in/out (20%)

    horizontal_flip=True,     # Randomly flip images horizontally

    fill_mode='nearest'       # Fill empty pixels after transformations
)

train_ds = train_gen.flow_from_directory(
    train_path,
    target_size=(150, 150),
    batch_size=32,
    class_mode = 'categorical'
)

In [ ]:
learning_rate = 0.001

optimizer = keras.optimizers.Adam(learning_rate=learning_rate)

loss = keras.losses.CategoricalCrossentropy()

model.compile(loss=loss, optimizer=optimizer, metrics=['accuracy'])

In [ ]:
res = model.fit(train_ds, epochs=10, validation_data=val_ds)

In [ ]:
plt.plot(res.history['accuracy'], label='train_acc')
plt.plot(res.history['val_accuracy'], label='val_acc')
plt.xticks(np.arange(10))
plt.legend()

# Adding more layers
- Adding one inner dense layer
- Experimenting with different sizes of inner layer

In [ ]:
def make_model(learning_rate=0.001, size_inner=100):
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(150, 150, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(150, 150, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)

    inner = keras.layers.Dense(size_inner, activation='relu')(vectors)

    outputs = keras.layers.Dense(10, activation='softmax')(inner)

    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy()

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
scores = {}

for size in [100, 256]:
    print(size)

    model = make_model(size_inner=size)
    history = model.fit(train_ds, epochs=10, validation_data=val_ds)
    scores[size] = history.history

    print()
    print()

In [ ]:
plt.plot(scores[256]['accuracy'], label='train_acc')
plt.plot(scores[256]['val_accuracy'], label='val_acc')
plt.xticks(np.arange(20))

# Dropout
- Regularizing by freezing a part of the network
- Adding dropout to our model
- Experimenting with different values

In [ ]:
def make_model(learning_rate=0.001, droprate=0.5):
    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(150, 150, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(150, 150, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)
    x = keras.layers.BatchNormalization()(vectors)

    drop = keras.layers.Dropout(droprate)(x)

    outputs = keras.layers.Dense(10, activation='softmax')(drop)

    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy()

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
scores = {}
for droprate in [0.2, 0.5]:
    print(droprate)

    model = make_model(
        droprate=droprate
    )

    history = model.fit(train_ds, epochs=10, validation_data=val_ds)
    scores[droprate] = history.history

    print()
    print()

In [ ]:
for size, hist in scores.items():
    plt.plot(hist['val_accuracy'], label=('val=%s' % size))

plt.xticks(np.arange(10))
plt.legend()

# Training a larger model
- Train a 224x224 model

In [ ]:
def make_model(input_size=150, learning_rate=0.01,
               droprate=0.5):

    base_model = VGG16(
        weights='imagenet',
        include_top=False,
        input_shape=(input_size, input_size, 3)
    )

    base_model.trainable = False

    #########################################

    inputs = keras.Input(shape=(input_size, input_size, 3))
    base = base_model(inputs, training=False)
    vectors = keras.layers.GlobalAveragePooling2D()(base)


    drop = keras.layers.Dropout(droprate)(vectors)
    outputs = keras.layers.Dense(10)(drop)

    model = keras.Model(inputs, outputs)

    #########################################

    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    loss = keras.losses.CategoricalCrossentropy(from_logits=True)

    model.compile(
        optimizer=optimizer,
        loss=loss,
        metrics=['accuracy']
    )

    return model

In [ ]:
input_size = 224

train_gen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    shear_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

train_ds = train_gen.flow_from_directory(
    train_path,
    target_size=(input_size, input_size),
    batch_size=32
)


val_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

val_ds = val_gen.flow_from_directory(
    val_path,
    target_size=(input_size, input_size),
    batch_size=32,
    shuffle=False
)

In [ ]:
checkpoint = keras.callbacks.ModelCheckpoint(
    'vgg16_v4_1_{epoch:02d}_{val_accuracy:.3f}.h5',
    save_best_only=True,
    monitor='val_accuracy',
    mode='max'
)

In [ ]:
learning_rate = 0.0005
droprate = 0.5

model = make_model(
    input_size=input_size,
    learning_rate=learning_rate,
    droprate=droprate
)

history = model.fit(train_ds, epochs=50, validation_data=val_ds,
                   callbacks=[checkpoint])

# Testing the *model*

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img
from tensorflow.keras.applications.vgg16 import preprocess_input

In [ ]:
test_path = '/content/clothing-dataset-small-master/test'

test_gen = ImageDataGenerator(preprocessing_function=preprocess_input)

test_ds = test_gen.flow_from_directory(
    test_path,
    target_size=(224, 224),
    batch_size=32,
    shuffle=False
)

In [ ]:
model = keras.models.load_model('vgg16_v4_1_08_0.84.h5')

In [ ]:
model.evaluate(test_ds)